# 20.16 — Streaming & real-time ML

Streaming ML turns event time into continuously updated features and decisions, while watermarks decide when late data is still allowed to matter.

Feature stores supply the transformation contract, while online inference supplies the freshness requirement. Streaming features then feed monitoring, real-time serving, and LLMOps guardrails. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 17
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

## The concept, built once

Watermark and streaming feature update: $w=t_{max}-\ell$ and $x_t=\alpha v_t+(1-\alpha)x_{t-1}$.

A real streaming system separates event time from arrival time. The reusable method below computes event-time windows, a watermark, late-event flags, exponential streaming features, and queue-induced freshness lag.

In [ ]:
def stream_windows(events, window_minutes=5, lateness_minutes=10, alpha=0.6, capacity_per_minute=1200):
    event_time = events[:, 0]
    arrival_time = events[:, 1]
    value = events[:, 2]
    t_max = np.max(event_time)
    watermark = t_max - lateness_minutes
    window_id = np.floor(event_time / window_minutes).astype(int)
    unique_windows = np.arange(np.min(window_id), np.max(window_id) + 1)
    counts = np.array([np.sum(window_id == w) for w in unique_windows])
    late = event_time < watermark
    order = np.argsort(arrival_time)
    sorted_arrival = arrival_time[order]
    sorted_event = event_time[order]
    sorted_value = value[order]
    smoothed = np.empty_like(sorted_value, dtype=float)
    previous = sorted_value[0]
    for i, current in enumerate(sorted_value):
        previous = alpha * current + (1 - alpha) * previous
        smoothed[i] = previous
    service_minutes = 1 / capacity_per_minute
    finish = np.empty_like(sorted_arrival, dtype=float)
    for i, arrival in enumerate(sorted_arrival):
        if i == 0:
            start = arrival
        else:
            start = max(arrival, finish[i - 1])
        finish[i] = start + service_minutes
    freshness_lag = finish - sorted_event
    p95_freshness_lag = float(np.quantile(freshness_lag, 0.95))
    throughput = min(capacity_per_minute, len(events) / max(np.ptp(arrival_time), 1))
    return {
        "watermark": float(watermark),
        "windows": unique_windows,
        "counts": counts,
        "late": late,
        "smoothed": smoothed,
        "freshness_lag": freshness_lag,
        "p95_freshness_lag": p95_freshness_lag,
        "throughput": float(throughput),
    }

lesson_events = np.array([
    [1, 1.0, 7],
    [3, 3.0, 10],
    [7, 7.0, 8],
    [88, 101.0, 6],
    [99, 99.0, 9],
    [100, 100.0, 11],
], dtype=float)

lesson_result = stream_windows(lesson_events, window_minutes=5, lateness_minutes=10, alpha=0.6)
lesson_counts = dict(zip(lesson_result["windows"], lesson_result["counts"]))
assert lesson_counts[0] == 2
assert lesson_counts[1] == 1
assert lesson_result["watermark"] == 90
assert bool(lesson_events[:, 0][3] < lesson_result["watermark"])
assert abs((0.6 * 10 + 0.4 * 7) - 8.8) < 1e-12
assert 1200 - 1000 == 200
print("window [0,5) count", lesson_counts[0])
print("window [5,10) count", lesson_counts[1])
print("watermark", lesson_result["watermark"])
print("event 88 late", bool(lesson_events[:, 0][3] < lesson_result["watermark"]))

The same method now becomes the single evaluation API. It returns the operational artifact and the topic metric so the D1 proof scales to D2–D5.

In [ ]:
print("Build-once assertions passed for 20.16")

## The dataset ladder

Family F17 uses an inline D1–D5 systems workload ladder. Each rung increases operational realism while keeping the computation CPU-only, seeded, and NumPy based.

In [ ]:
def build_streaming_ladder(seed=17):
    rng = np.random.default_rng(seed)
    ladder = []
    d1 = lesson_events.copy()
    ladder.append({"name": "D1 hand events", "events": d1, "capacity": 1200, "rate": 700})
    n = 100000
    arrival = np.cumsum(rng.exponential(1 / 900, size=n))
    event = arrival - rng.exponential(0.05, size=n)
    value = rng.normal(10, 2, size=n)
    d2 = np.column_stack([event, arrival, value])
    ladder.append({"name": "D2 clean ordered 100k", "events": d2, "capacity": 1200, "rate": 900})
    arrival = np.cumsum(rng.exponential(1 / 1200, size=n))
    late_delay = rng.choice([0, 2, 12], size=n, p=[0.94, 0.04, 0.02])
    event = arrival - late_delay - rng.exponential(0.03, size=n)
    value = rng.normal(10, 3, size=n)
    burst = rng.choice(n, size=n // 20, replace=False)
    arrival[burst] = arrival[burst] * 0.995
    d3 = np.column_stack([event, arrival, value])
    ladder.append({"name": "D3 late bursts", "events": d3, "capacity": 1100, "rate": 1200})
    n4 = 140000
    regimes = rng.choice([500, 900, 1500], size=n4, p=[0.35, 0.45, 0.20])
    arrival = np.cumsum(rng.exponential(1 / regimes))
    event = arrival - rng.gamma(1.5, 0.08, size=n4)
    value = 8 + 3 * np.sin(arrival / 20) + rng.normal(0, 1.5, size=n4)
    d4 = np.column_stack([event, arrival, value])
    ladder.append({"name": "D4 real-ish Poisson trace", "events": d4, "capacity": 1300, "rate": 1300})
    n5 = 180000
    arrival = np.cumsum(rng.exponential(1 / 1500, size=n5))
    overload = arrival > np.quantile(arrival, 0.55)
    arrival[overload] = arrival[overload] * 0.75
    event = arrival - rng.gamma(2.0, 0.12, size=n5)
    value = rng.normal(12, 4, size=n5)
    d5 = np.column_stack([event, arrival, value])
    ladder.append({"name": "D5 production backpressure", "events": d5, "capacity": 1000, "rate": 1500})
    return ladder

ladder = build_streaming_ladder()
for rung in ladder:
    events = rung["events"]
    print(rung["name"], "shape", events.shape, "capacity", rung["capacity"], "sample", np.round(events[:3], 3).tolist())

## Run the same method across D1–D5

The metric is collected in the same way for every rung, so changes across the ladder reflect workload complexity rather than a different measurement.

In [ ]:
results = []
for rung in ladder:
    result = stream_windows(rung["events"], capacity_per_minute=rung["capacity"])
    results.append({"name": rung["name"], "metric": result["p95_freshness_lag"], "artifact": result, "load": rung["rate"], "capacity": rung["capacity"]})

print("rung | p95 freshness lag | input rate | capacity")
for item in results:
    print(item["name"], round(item["metric"], 4), item["load"], item["capacity"])

## Results visualization

The closing figure has operational panels for each rung plus a metric-vs-load curve.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, item in enumerate(results):
    artifact = item["artifact"]
    counts = artifact["counts"][:40]
    axes[0, col].plot(counts, color="tab:blue")
    axes[0, col].set_title(item["name"].split()[0] + " windows")
    axes[0, col].set_xlabel("window")
    axes[0, col].set_ylabel("events")
    lag_sample = artifact["freshness_lag"][:1000]
    axes[1, col].hist(lag_sample, bins=30, color="tab:orange")
    axes[1, col].set_xlabel("freshness lag")
    axes[1, col].set_ylabel("count")
fig.suptitle("Operational panels: window load and freshness lag by rung")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(7, 4))
loads = [item["load"] for item in results]
metrics = [item["metric"] for item in results]
ax.plot(loads, metrics, marker="o")
ax.set_xlabel("input rate")
ax.set_ylabel("p95 freshness lag")
ax.set_title("Metric vs load")
plt.show()

## Pitfall on the hardest rung

Reproduce the lesson pitfall on D5, then apply the operational fix and compare the metric.

In [ ]:
d5 = ladder[-1]
ignored = stream_windows(d5["events"], capacity_per_minute=d5["capacity"])
scaled = stream_windows(d5["events"], capacity_per_minute=1800)
print("wrong capacity", d5["capacity"], "p95 lag", round(ignored["p95_freshness_lag"], 4))
print("fixed capacity", 1800, "p95 lag", round(scaled["p95_freshness_lag"], 4))
print("improvement", round(ignored["p95_freshness_lag"] - scaled["p95_freshness_lag"], 4))

## Evaluate it + Practice

- Metric: p95 freshness lag, compared with a no-watermark processing-time baseline.
- Sanity check: D1 must keep counts 2 and 1 and watermark 90.
- Ablation: lower capacity below input rate and verify p95 lag rises.
- Failure signals: monotonically growing lag, many late events, and stale feature panels.

Practice prompts:
1. Change one workload knob on D3 and predict whether the metric should improve or degrade.
2. Add one slice or route to D4 and explain which operational panel should change.
3. Tighten the D5 budget or threshold and rerun only after saving your own copy.